# Week 5 (update 6): alert threshold, top features and retraining on the relabelled data

COMP6310 group project, 6 November 2026. This re-runs the Week 5 questions of `ddos_week5_threshold_retraining_update5.ipynb`
(5 Nov 2026) on the **app's state after the 20 Feb relabelling and the retrain of every `f7ip` model**. What changed since
update 5:

- **Data.** The published 20 Feb file is replaced by `data\Tuesday-20-02-2018_relabelled.csv`
  (`scripts\pcap\relabel_feb20.py`): 797 UDP flood flows that the published file labels LOIC-HTTP are the 20 Feb
  **LOIC-UDP** flood (13:14–13:29), 702 of the victim's ICMP port-unreachable replies and 130 direction-flipped LOIC-HTTP
  flows that it labels Benign are attack by-products; 1,629 of 7,948,748 rows change, nothing else. Timestamps are
  24-hour. The 21 Feb file is the capture rebuilt from the raw packets, as in update 5. LOIC-UDP therefore has 4,827 rows
  across both days (2,527 flood flows + 2,300 ICMP reply rows) instead of 3,328.
- **Models.** All 42 `f7ip` models were retrained on these files on 5 Nov 2026 (`docs\retrain_on_relabelled_feb20_2026-11-05.md`).
- **Threshold.** The app now starts each model at its own default (`model_store.default_threshold`): the lowest tested
  threshold whose false-alarm rate on the model's benign test split is within 0.1 %, else 0.5. Chosen from the benign
  side only, so a held-out tool never sets it.
- **Feature numbers** are the app's current catalogue: F1 flows to the same destination IP (`dst_ip_flows_10s`),
  F2 distinct source IPs to the same destination IP (`dst_ip_src_ips_10s`), F3 flows from the same source IP
  (`src_ip_flows_10s`), F4 forward/backward packet ratio (`fwd_bwd_ratio`), F5 no-reply and empty-payload indicators,
  F6 timing regularity (`iat_cv`). Seven columns, 10-second look-back windows built on every row of each file.

Everything goes through the app's own code (`app/core/features.py`, `train/train_model.py`, `app/core/model_store.py`),
so the splits, features and the threshold rule are exactly the app's. The group's decisions kept: an **alert** is the
model's flag at a threshold, retraining is shown by a demonstration, and the three model types (XGBoost main, Random
Forest, linear SVM) are compared on the same splits.

## 0. Imports (the app's modules)

In [1]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path("../DDoSDetector").resolve()))
from app.core.features import ATTACK_TOOLS, FEATURES_F7IP, TIME, summary
from app.core.model_store import (FALLBACK_THRESHOLD, FPR_BUDGET, default_threshold, list_models, load_bundle,
                                  read_meta)
from app.core.models import ACCEPTS_MISSING, MODEL_TYPES, NAME_SUFFIX, XGBOOST, make_model
from app.paths import DATA_DIR, MODELS_DIR
from train.train_model import FILES_IP, drop_unscorable, load, split_all_tools, split_held_out

pd.set_option("display.width", 220, "display.max_columns", 30)
COLS = FEATURES_F7IP
print(sys.executable)
print(summary(COLS), COLS)
print("training files:", FILES_IP, "| false-alarm budget for the default threshold:", FPR_BUDGET)

C:\Users\Xavie\source\repos\DDOS-ML-Detection\.venv\Scripts\python.exe
6 feature groups (7 columns) ['dst_ip_flows_10s', 'dst_ip_src_ips_10s', 'src_ip_flows_10s', 'fwd_bwd_ratio', 'no_reply', 'empty_payload', 'iat_cv']
training files: ['Tuesday-20-02-2018_relabelled.csv', 'Wednesday-21-02-2018_victim_capture_labelled.csv'] | false-alarm budget for the default threshold: 0.001


## 1. Load the two files exactly as training does (window features built per file on every row)

In [2]:
t0 = time.time()
df = load(DATA_DIR, None, FILES_IP)
df = drop_unscorable(df, COLS)
print(f"{len(df):,} flows in {time.time() - t0:.0f} s")
print(df["tool"].value_counts().to_string())
print("\nLOIC-UDP by file and label reason (the 20 Feb flood was labelled LOIC-HTTP in the published file):")
print(df[df["tool"] == "LOIC-UDP"].groupby("source_file").size().to_string())
print("\nmedians per file and tool:")
print(df.groupby(["source_file", "tool"])[COLS].median().round(2).to_string())

  Tuesday-20-02-2018_relabelled.csv: 7,948,748 benign/DDoS rows (60s)


   optional column missing in Wednesday-21-02-2018_victim_capture_labelled.csv (left as missing values): Active Mean
   optional column missing in Wednesday-21-02-2018_victim_capture_labelled.csv (left as missing values): Idle Mean
   optional column missing in Wednesday-21-02-2018_victim_capture_labelled.csv (left as missing values): Fwd Act Data Pkts
  Wednesday-21-02-2018_victim_capture_labelled.csv: 2,169,917 benign/DDoS rows (9s)


  cleaning: 10,118,665 -> 9,880,087 rows


9,880,087 flows in 132 s
tool
Benign       7182913
HOIC         2116841
LOIC-HTTP     575506
LOIC-UDP        4827

LOIC-UDP by file and label reason (the 20 Feb flood was labelled LOIC-HTTP in the published file):


source_file
Tuesday-20-02-2018_relabelled.csv                   1499
Wednesday-21-02-2018_victim_capture_labelled.csv    3328

medians per file and tool:


                                                            dst_ip_flows_10s  dst_ip_src_ips_10s  src_ip_flows_10s  fwd_bwd_ratio  no_reply  empty_payload  iat_cv
source_file                                      tool                                                                                                             
Tuesday-20-02-2018_relabelled.csv                Benign                  8.0                 4.0               8.0           0.89       0.0            0.0    0.85
                                                 LOIC-HTTP            1530.0                10.0             154.0           0.60       0.0            0.0    2.44
                                                 LOIC-UDP                5.0                 1.0               7.0        1860.00       1.0            0.0    1.38
Wednesday-21-02-2018_victim_capture_labelled.csv Benign                  0.0                 0.0               0.0           1.00       0.0            0.0    0.86
                      

## 1b. Where the benign flows sit relative to the attacks (why the benign splits differ)

The held-out models come in three benign splits. `per_day` tests on the later 30 % of each file's benign flows,
`by_file` on the later file's benign flows only (the rebuilt 21 Feb capture has 2,002 genuinely benign flows) and
`attack_period` on the benign flows inside the held-out tool's attack window. With window features, benign flows on
the attacked network *during* an attack carry high counts too, so it matters which split puts them in training.
New since update 5: the 20 Feb stamps are now 24-hour (the published file's "01:00–07:59" was the afternoon), which
changes which benign rows are "the earlier 70 %" of 20 Feb; and LOIC-UDP now has a run on each day, so its
"attack window" across the two files spans the 20 Feb afternoon and the 21 Feb morning.

In [3]:
ben = df[df["tool"] == "Benign"]
for f, g in df.groupby("source_file"):
    b = g[g["tool"] == "Benign"]
    t70 = b[TIME].quantile(0.7)
    print(f"{f}: {len(b):,} benign flows {b[TIME].min()} .. {b[TIME].max()}; per_day training benign end at {t70}")
    for tool, a in g[g["tool"] != "Benign"].groupby("tool"):
        lo, hi = a[TIME].min(), a[TIME].max()
        inside = b[b[TIME].between(lo, hi)]
        victim = a["Dst IP"].mode().iat[0]
        to_victim = inside[inside["Dst IP"] == victim]
        print(f"  {tool}: {len(a):,} flows {lo} .. {hi} (victim {victim}); benign flows inside the window: "
              f"{len(inside):,}, of which to the victim {len(to_victim):,}; "
              f"{'before' if hi <= t70 else 'after' if lo >= t70 else 'across'} the per_day 70 % point")
        if len(inside):
            print("    benign-inside medians:", inside[COLS[:3]].median().round(1).to_dict(),
                  "| to-victim medians:", to_victim[COLS[:3]].median().round(1).to_dict() if len(to_victim) else "-")

Tuesday-20-02-2018_relabelled.csv: 7,180,931 benign flows 2018-02-20 08:00:00 .. 2018-02-20 19:59:59; per_day training benign end at 2018-02-20 14:42:07


  LOIC-HTTP: 575,506 flows 2018-02-20 10:13:54 .. 2018-02-20 11:16:48 (victim 172.31.69.25); benign flows inside the window: 900,104, of which to the victim 135; before the per_day 70 % point
    benign-inside medians: {'dst_ip_flows_10s': 8.0, 'dst_ip_src_ips_10s': 4.0, 'src_ip_flows_10s': 8.0} | to-victim medians: {'dst_ip_flows_10s': 1531.0, 'dst_ip_src_ips_10s': 10.0, 'src_ip_flows_10s': 0.0}
  LOIC-UDP: 1,499 flows 2018-02-20 13:14:17 .. 2018-02-20 13:29:15 (victim 172.31.69.25); benign flows inside the window: 196,143, of which to the victim 14; before the per_day 70 % point
    benign-inside medians: {'dst_ip_flows_10s': 8.0, 'dst_ip_src_ips_10s': 4.0, 'src_ip_flows_10s': 8.0} | to-victim medians: {'dst_ip_flows_10s': 13.0, 'dst_ip_src_ips_10s': 3.0, 'src_ip_flows_10s': 0.0}


Wednesday-21-02-2018_victim_capture_labelled.csv: 1,982 benign flows 2018-02-21 08:32:54 .. 2018-02-21 17:29:19; per_day training benign end at 2018-02-21 15:40:48.900000


  HOIC: 2,116,841 flows 2018-02-21 14:11:08 .. 2018-02-21 15:05:54 (victim 172.31.69.28); benign flows inside the window: 474, of which to the victim 291; before the per_day 70 % point
    benign-inside medians: {'dst_ip_flows_10s': 4836.0, 'dst_ip_src_ips_10s': 10.0, 'src_ip_flows_10s': 0.0} | to-victim medians: {'dst_ip_flows_10s': 6998.0, 'dst_ip_src_ips_10s': 11.0, 'src_ip_flows_10s': 0.0}
  LOIC-UDP: 3,328 flows 2018-02-21 10:08:51 .. 2018-02-21 10:43:21 (victim 172.31.69.28); benign flows inside the window: 73, of which to the victim 52; before the per_day 70 % point
    benign-inside medians: {'dst_ip_flows_10s': 2.0, 'dst_ip_src_ips_10s': 1.0, 'src_ip_flows_10s': 0.0} | to-victim medians: {'dst_ip_flows_10s': 5.5, 'dst_ip_src_ips_10s': 2.0, 'src_ip_flows_10s': 0.0}


## 2. The alert threshold, measured on the app's models

A flow raises an alert when its model score is at or above the threshold. Each of the app's `f7ip` models (seed 0) is
scored on its own test split, rebuilt here as the training script built it; the first check is that recall at 0.5
equals the number stored with the model. The linear SVM cannot score a row with a missing value; such rows count as
not flagged, as in the app.

"All tools" is the optimistic random split; "held out" means the tool was not in training. Two things to look for:
the held-out LOIC-HTTP scores (XGBoost ranks the attack correctly but scores it below 0.5), and held-out LOIC-UDP,
which update 5 reported as 0.52 "with a ceiling": that number was not a held-out result, because the 797 LOIC-UDP flood
flows of 20 Feb sat in the training set under the LOIC-HTTP label.

In [4]:
THRESHOLDS = [0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.01, 0.001]
CASES = {"all tools (random split)": ("all_tools_f7ip", None),
         "HOIC held out": ("hoic_held_out_f7ip", "HOIC"),
         "LOIC-HTTP held out": ("loic_http_held_out_f7ip", "LOIC-HTTP"),
         "LOIC-UDP held out": ("loic_udp_held_out_f7ip", "LOIC-UDP")}

def scores_of(bundle, te):
    X = te[COLS]
    ok = np.ones(len(X), bool) if ACCEPTS_MISSING[bundle.model_type] else np.isfinite(X.to_numpy(float)).all(axis=1)
    p = np.zeros(len(X))
    p[ok] = bundle.model.predict_proba(X[ok])[:, 1]
    return p

SPLITS, APP, rows = {}, {}, []
for label, (stem, held) in CASES.items():
    SPLITS[label] = split_all_tools(df, 0) if held is None else split_held_out(df, 0, held, "per_day")[:2]
    tr, te = SPLITS[label]
    y, tool = te["y"].to_numpy(), te["tool"].to_numpy()
    for kind in MODEL_TYPES:
        name = stem + NAME_SUFFIX[kind]
        b = load_bundle(MODELS_DIR / f"{name}.meta.json")
        p = scores_of(b, te)
        APP[(label, kind)] = (b, te, p)
        at05 = {t: round(float((p[tool == t] >= 0.5).mean()), 4) for t in b.meta["metrics"]["recall"]}
        same = all(abs(at05[t] - b.meta["metrics"]["recall"][t]) < 5e-4 for t in at05)
        print(f"{name:<32} recall at 0.5 here {at05} | stored {b.meta['metrics']['recall']} | same: {same} | "
              f"trained on {b.meta['data_files'][0]}")
        for thr in THRESHOLDS:
            pred = p >= thr
            row = {"case": label, "model": MODEL_TYPES[kind], "threshold": thr,
                   "false alarms": int(pred[y == 0].sum()), "FPR": round(float(pred[y == 0].mean()), 5)}
            for t in ATTACK_TOOLS:
                if (tool == t).any():
                    row[f"recall {t}"] = round(float(pred[tool == t].mean()), 4)
            rows.append(row)
thr_table = pd.DataFrame(rows)
for label in CASES:
    te = SPLITS[label][1]
    print(f"\n=== {label} ({int((te['y'] == 0).sum()):,} benign test flows) ===")
    g = thr_table[thr_table["case"] == label].drop(columns="case").dropna(axis=1, how="all")
    print(g.pivot(index="threshold", columns="model").sort_index(ascending=False).to_string())

all_tools_f7ip                   recall at 0.5 here {'HOIC': 1.0, 'LOIC-HTTP': 0.9999, 'LOIC-UDP': 0.989} | stored {'HOIC': 1.0, 'LOIC-HTTP': 0.9999, 'LOIC-UDP': 0.989} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


all_tools_f7ip_rf                recall at 0.5 here {'HOIC': 0.9999, 'LOIC-HTTP': 0.9996, 'LOIC-UDP': 0.9731} | stored {'HOIC': 0.9999, 'LOIC-HTTP': 0.9996, 'LOIC-UDP': 0.9731} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


all_tools_f7ip_svm               recall at 0.5 here {'HOIC': 0.9999, 'LOIC-HTTP': 0.9994, 'LOIC-UDP': 0.4938} | stored {'HOIC': 0.9999, 'LOIC-HTTP': 0.9994, 'LOIC-UDP': 0.4938} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


hoic_held_out_f7ip               recall at 0.5 here {'HOIC': 0.9999} | stored {'HOIC': 0.9999} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


hoic_held_out_f7ip_rf            recall at 0.5 here {'HOIC': 0.7552} | stored {'HOIC': 0.7552} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


hoic_held_out_f7ip_svm           recall at 0.5 here {'HOIC': 0.9999} | stored {'HOIC': 0.9999} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


... (3 outputs left out)

loic_udp_held_out_f7ip           recall at 0.5 here {'LOIC-UDP': 0.0} | stored {'LOIC-UDP': 0.0} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


loic_udp_held_out_f7ip_rf        recall at 0.5 here {'LOIC-UDP': 0.0} | stored {'LOIC-UDP': 0.0} | same: True | trained on Tuesday-20-02-2018_relabelled.csv


loic_udp_held_out_f7ip_svm       recall at 0.5 here {'LOIC-UDP': 0.0} | stored {'LOIC-UDP': 0.0} | same: True | trained on Tuesday-20-02-2018_relabelled.csv



=== all tools (random split) (300,000 benign test flows) ===
          false alarms                              FPR                        recall HOIC                       recall LOIC-HTTP                       recall LOIC-UDP                      
model       Linear SVM Random Forest XGBoost Linear SVM Random Forest  XGBoost  Linear SVM Random Forest XGBoost       Linear SVM Random Forest XGBoost      Linear SVM Random Forest XGBoost
threshold                                                                                                                                                                                    
0.900                9             0       0    0.00003       0.00000  0.00000      0.8958        0.9993     1.0           0.0000        0.9982  0.9998          0.0000        0.7465  0.9827
0.700               16             0       2    0.00005       0.00000  0.00001      0.9997        0.9999     1.0           0.9853        0.9995  0.9999          0.3211        0.9

In [5]:
# How low do the scores of the attack flows that the held-out XGBoost models miss at 0.5 go?
for label in list(CASES)[1:]:
    b, te, p = APP[(label, XGBOOST)]
    tool = CASES[label][1]
    s = p[te["tool"].to_numpy() == tool]
    ben = p[te["tool"].to_numpy() == "Benign"]
    q = np.quantile(s, [0.01, 0.05, 0.1, 0.5, 0.9])
    print(f"{label}: {tool} score quantiles 1/5/10/50/90%: " + ", ".join(f"{v:.2e}" for v in q)
          + f" | missed at 0.5: {int((s < 0.5).sum()):,} of {len(s):,} | benign median {np.median(ben):.2e}, "
          f"benign 99.9th percentile {np.quantile(ben, 0.999):.2e}")

HOIC held out: HOIC score quantiles 1/5/10/50/90%: 1.00e+00, 1.00e+00, 1.00e+00, 1.00e+00, 1.00e+00 | missed at 0.5: 152 of 2,116,841 | benign median 1.02e-07, benign 99.9th percentile 1.93e-02


LOIC-HTTP held out: LOIC-HTTP score quantiles 1/5/10/50/90%: 2.09e-04, 6.50e-02, 6.50e-02, 9.59e-02, 3.98e-01 | missed at 0.5: 569,934 of 575,506 | benign median 2.57e-07, benign 99.9th percentile 4.71e-03
LOIC-UDP held out: LOIC-UDP score quantiles 1/5/10/50/90%: 8.71e-08, 1.65e-07, 1.79e-07, 8.27e-07, 4.90e-06 | missed at 0.5: 4,827 of 4,827 | benign median 2.32e-07, benign 99.9th percentile 1.37e-03


### 2b. One threshold does not fit every model: the app's per-model default

A model's scores are calibrated to the tools it was trained on. The rule the app uses since 6 Nov 2026
(`model_store.default_threshold`): take the model's own threshold sweep (0.5, 0.2, 0.1, 0.05, 0.01; recall and
false-alarm rate on its test split, written at training time) and pick the **lowest threshold whose false-alarm rate is
within 0.1 %**; 0.5 when none is. It uses the benign side only, so an unseen tool's scores never set it. Below, the rule
applied to every seed-0 `f7ip` model, with what it buys in recall.

In [6]:
rows = []
for p_ in sorted(list_models(MODELS_DIR)):
    meta = read_meta(p_)
    name = p_.name.removesuffix(".meta.json")
    if meta.get("feature_set") != "f7ip" or meta.get("seed", 0) != 0:
        continue
    thr, why = default_threshold(meta)
    sw = {s["threshold"]: s for s in meta["metrics"]["threshold_sweep"]}
    at = sw.get(thr, {})
    held = meta.get("held_out_tool")
    rows.append({"model": name, "held out": held or "-", "split": meta.get("benign_split_mode"),
                 "default": thr, "FPR at default": at.get("fpr"), "recall at default": at.get("recall"),
                 "recall at 0.5": sw[0.5]["recall"], "FPR at 0.5": sw[0.5]["fpr"],
                 "fallback": thr == FALLBACK_THRESHOLD and "no tested threshold" in why})
defaults = pd.DataFrame(rows).sort_values(["held out", "split", "model"])
print(defaults.to_string(index=False))
print("\nreason text for three models:")
for n in ("all_tools_f7ip", "loic_http_held_out_f7ip", "hoic_held_out_by_file_f7ip"):
    print(f"  {n}: {default_threshold(read_meta(MODELS_DIR / (n + '.meta.json')))[1]}")

                                    model  held out         split  default  FPR at default  recall at default  recall at 0.5  FPR at 0.5  fallback
                           all_tools_f7ip         -        random     0.01          0.0009             1.0000         0.9999      0.0000     False
                        all_tools_f7ip_rf         -        random     0.10          0.0005             0.9999         0.9998      0.0000     False
                       all_tools_f7ip_svm         -        random     0.50          0.0005             0.9989         0.9989      0.0005     False
         hoic_held_out_attack_period_f7ip      HOIC attack_period     0.50          0.0084             0.9999         0.9999      0.0084      True
      hoic_held_out_attack_period_f7ip_rf      HOIC attack_period     0.50          0.1498             0.9957         0.9957      0.1498      True
     hoic_held_out_attack_period_f7ip_svm      HOIC attack_period     0.50          0.4578             0.9999         

## 3. Top features (SHAP, XGBoost)

Mean absolute SHAP value of each input (how much it moves the score, in log-odds) on 2,000 test flows of the
all-tools model; then, for each held-out model, the average SHAP value per input for the held-out tool's flows it
catches and the ones it misses **at that model's own default threshold** (positive pushes towards DDoS).

In [7]:
import shap

def mean_abs_shap(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    return pd.Series(np.abs(sv).mean(axis=0), index=X.columns).sort_values(ascending=False)

b, te, p = APP[("all tools (random split)", XGBOOST)]
sample = te.sample(2000, random_state=0)
print("All-tools XGBoost, mean |SHAP| on 2,000 test flows (log-odds):")
print(mean_abs_shap(b.model, sample[COLS]).round(3).to_string())

for label in list(CASES)[1:]:
    b, te, p = APP[(label, XGBOOST)]
    tool = CASES[label][1]
    thr0 = default_threshold(b.meta)[0]
    att = te[te["tool"] == tool].assign(score=p[(te["tool"] == tool).to_numpy()])
    caught, missed = att[att["score"] >= thr0], att[att["score"] < thr0]
    explainer = shap.TreeExplainer(b.model)
    parts = {}
    for name, frame in (("caught", caught), ("missed", missed)):
        if len(frame):
            sub = frame.sample(min(1000, len(frame)), random_state=0)
            parts[f"{name} {tool} flows (n={len(frame):,})"] = explainer.shap_values(sub[COLS]).mean(axis=0)
            parts[f"{name}: median input"] = sub[COLS].median().to_numpy()
    print(f"\n{label} XGBoost model at its default threshold {thr0}: average SHAP value per input and median input values:")
    print(pd.DataFrame(parts, index=COLS).round(2).to_string())

C:\Users\Xavie\source\repos\DDOS-ML-Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All-tools XGBoost, mean |SHAP| on 2,000 test flows (log-odds):


dst_ip_flows_10s      12.007
dst_ip_src_ips_10s     1.961
fwd_bwd_ratio          1.096
src_ip_flows_10s       0.759
iat_cv                 0.684
no_reply               0.442
empty_payload          0.230



HOIC held out XGBoost model at its default threshold 0.05: average SHAP value per input and median input values:
                    caught HOIC flows (n=2,116,761)  caught: median input  missed HOIC flows (n=80)  missed: median input
dst_ip_flows_10s                              11.99                7069.0                     -5.67                  20.0
dst_ip_src_ips_10s                            -1.04                  11.0                      1.77                   1.0
src_ip_flows_10s                              -0.70                 710.0                     -1.87                  10.5
fwd_bwd_ratio                                  0.66                   2.0                      1.07                   2.0
no_reply                                      -0.08                   1.0                      0.05                   1.0
empty_payload                                 -0.05                   1.0                     -0.44                   1.0
iat_cv                          


LOIC-HTTP held out XGBoost model at its default threshold 0.01: average SHAP value per input and median input values:
                    caught LOIC-HTTP flows (n=552,763)  caught: median input  missed LOIC-HTTP flows (n=22,743)  missed: median input
dst_ip_flows_10s                                  0.08               1531.00                              -2.60               1530.00
dst_ip_src_ips_10s                               -1.24                 10.00                              -4.44                 12.00
src_ip_flows_10s                                 -1.04                154.00                              -1.02                153.00
fwd_bwd_ratio                                     0.91                  0.60                               0.88                  0.60
no_reply                                          0.52                  0.00                               0.48                  0.00
empty_payload                                    -0.15                  0.00 


LOIC-UDP held out XGBoost model at its default threshold 0.01: average SHAP value per input and median input values:
                    missed LOIC-UDP flows (n=4,827)  missed: median input
dst_ip_flows_10s                              -8.80                  5.00
dst_ip_src_ips_10s                            -0.26                  1.00
src_ip_flows_10s                              -2.12                  6.00
fwd_bwd_ratio                                 -1.69              33792.00
no_reply                                       0.05                  1.00
empty_payload                                  0.28                  0.00
iat_cv                                        -1.24                 19.02


### 3b. Why unseen LOIC-UDP is 0.000 and unseen LOIC-HTTP sits below 0.5

The XGBoost held-out models' split points on the three window features, against where each tool and the benign
traffic sit. The LOIC-HTTP-held-out model learned HOIC's size (about 7,000 flows per 10 s, 700 per sender, 12 or more
senders); LOIC-HTTP's 1,500 flows, 150 per sender and 10 senders fall below those splits, in a range that part of the
20 Feb whole-network benign traffic also occupies. The LOIC-UDP-held-out model never saw a flow with no reply at all
and an extreme ratio from a sender with tiny window counts.

In [8]:
import re
for label in ("LOIC-HTTP held out", "LOIC-UDP held out"):
    b = APP[(label, XGBOOST)][0]
    dump = b.model.get_booster().get_dump()
    print(f"\n{label} XGBoost ({len(dump)} trees): split thresholds used, 5th/25th/50th/75th/95th percentile")
    for c in COLS:
        th = np.array(sorted(float(v) for tree in dump for v in re.findall(rf"\[{re.escape(c)}<([0-9.eE+-]+)\]", tree)))
        if len(th):
            print(f"  {c:<20} {len(th):>4} splits: {np.percentile(th, [5, 25, 50, 75, 95]).round(2)}")
print("\nwhere the tools and the benign traffic sit (median / 99th percentile, per file):")
q = df.groupby(["source_file", "tool"])[COLS[:3]].quantile([0.5, 0.99]).unstack().round(1)
print(q.to_string())


LOIC-HTTP held out XGBoost (300 trees): split thresholds used, 5th/25th/50th/75th/95th percentile
  dst_ip_flows_10s      996 splits: [1.000e+00 2.000e+00 1.090e+02 1.366e+03 3.961e+03]
  dst_ip_src_ips_10s    386 splits: [ 2.  2.  6. 12. 12.]
  src_ip_flows_10s     1068 splits: [  1.   3.   8.  29. 436.]
  fwd_bwd_ratio         776 splits: [ 0.56  2.33  5.   10.   72.  ]
  no_reply              102 splits: [1. 1. 1. 1. 1.]
  empty_payload         205 splits: [1. 1. 1. 1. 1.]
  iat_cv               1170 splits: [0.11 0.69 1.19 1.46 2.25]

LOIC-UDP held out XGBoost (300 trees): split thresholds used, 5th/25th/50th/75th/95th percentile
  dst_ip_flows_10s     1065 splits: [1.000e+00 1.500e+01 8.300e+01 2.820e+02 1.489e+03]
  dst_ip_src_ips_10s    701 splits: [ 2.  3. 10. 13. 16.]
  src_ip_flows_10s     1043 splits: [  1.   2.   8.  37. 108.]
  fwd_bwd_ratio         457 splits: [0.5  0.6  0.6  1.51 2.11]
  no_reply               38 splits: [1. 1. 1. 1. 1.]
  empty_payload         117 spli

                                                           dst_ip_flows_10s         dst_ip_src_ips_10s        src_ip_flows_10s       
                                                                       0.50    0.99               0.50   0.99             0.50   0.99
source_file                                      tool                                                                                
Tuesday-20-02-2018_relabelled.csv                Benign                 8.0  1631.0                4.0  122.0              8.0  353.0
                                                 LOIC-HTTP           1530.0  1604.0               10.0   12.0            154.0  164.0
                                                 LOIC-UDP               5.0    24.0                1.0    5.0              7.0   20.0
Wednesday-21-02-2018_victim_capture_labelled.csv Benign                 0.0  7993.0                0.0   12.0              0.0    6.2
                                                 HOIC         

## 4. Retraining demonstration: a new attack tool appears

Same scenario as in updates 4 and 5: the model was trained without one tool (the held-out setup); that tool attacks;
analysts label the **first flows of the attack** (the first 100, 1,000 or 10,000 by start time) and XGBoost is retrained
with them (the training script's `--extra-labelled` mechanism, here the same rows added in code). The model is tested on
the **rest of the attack** (every flow after the first 10,000) and the same benign test flows. "Whole incident" = the
first flows labelled whether flagged or not; "alerts only" = only the flows the old model flagged (at 0.5) among them.
A **quick update** adds 50 trees to the old model on 1,000 whole-incident flows plus 100,000 rows of the original
training data. Seeds 0 and 1; the app's settings (300 trees, depth 6, learning rate 0.1, class weights).

Because the held-out LOIC-HTTP scores sit below 0.5, results are given at **both 0.5 and 0.01** (the app's default for
these models).

In [9]:
def fit(train, seed, base=None, n_estimators=300):
    spw = (train["y"] == 0).sum() / (train["y"] == 1).sum()
    m, _ = make_model(XGBOOST, spw, seed)
    m.set_params(n_estimators=n_estimators)
    t1 = time.perf_counter()
    m.fit(train[COLS], train["y"], xgb_model=None if base is None else base.get_booster())
    return m, time.perf_counter() - t1

def score_on(model, te_ben, te_att):
    pa = model.predict_proba(te_att[COLS])[:, 1]
    pb = model.predict_proba(te_ben[COLS])[:, 1]
    out = {}
    for thr in (0.5, 0.01):
        out[f"recall@{thr}"] = round(float((pa >= thr).mean()), 4)
        out[f"false alarms@{thr}"] = int((pb >= thr).sum())
    return out

demo = []
for tool in ["HOIC", "LOIC-HTTP"]:
    for seed in [0, 1]:
        tr, te, _ = split_held_out(df, seed, tool, "per_day")
        att = te[te["tool"] == tool].sort_values(TIME, kind="stable")
        pool, rest = att.iloc[:10_000], att.iloc[10_000:]
        te_ben = te[te["tool"] == "Benign"]
        base, secs = fit(tr, seed)
        flagged = pool[base.predict_proba(pool[COLS])[:, 1] >= 0.5]
        info = {"tool": tool, "seed": seed, "of first 10,000 flagged at 0.5": len(flagged), "rest of attack": len(rest)}
        demo.append({**info, "model": "before (no labels)", "labelled flows": 0, "train s": round(secs, 1),
                     **score_on(base, te_ben, rest)})
        for source, frame in [("whole incident", pool), ("alerts only", flagged)]:
            used = 0
            for n in [100, 1000, 10000]:
                fb = frame.iloc[:n]
                if len(fb) == used:
                    continue
                used = len(fb)
                m, secs = fit(pd.concat([tr, fb]), seed)
                demo.append({**info, "model": f"retrained, {source}", "labelled flows": len(fb),
                             "train s": round(secs, 1), **score_on(m, te_ben, rest)})
        fb = pool.iloc[:1000]
        upd = pd.concat([tr.sample(100_000, random_state=seed), fb])
        m, secs = fit(upd, seed, base=base, n_estimators=50)
        demo.append({**info, "model": "quick update (+50 trees), whole incident", "labelled flows": len(fb),
                     "train s": round(secs, 1), **score_on(m, te_ben, rest)})
        span = pool[TIME].max() - pool[TIME].min()
        print(f"{tool} seed {seed} done: the first 10,000 flows span {span}; {len(flagged):,} of them were flagged at 0.5; "
              f"{len(te_ben):,} benign test flows")
demo_table = pd.DataFrame(demo)
print(demo_table.to_string(index=False))

HOIC seed 0 done: the first 10,000 flows span 0 days 00:00:17; 9,854 of them were flagged at 0.5; 2,154,875 benign test flows


HOIC seed 1 done: the first 10,000 flows span 0 days 00:00:17; 9,871 of them were flagged at 0.5; 2,154,875 benign test flows


LOIC-HTTP seed 0 done: the first 10,000 flows span 0 days 00:01:11; 4,498 of them were flagged at 0.5; 2,154,875 benign test flows


LOIC-HTTP seed 1 done: the first 10,000 flows span 0 days 00:01:11; 3,766 of them were flagged at 0.5; 2,154,875 benign test flows
     tool  seed  of first 10,000 flagged at 0.5  rest of attack                                    model  labelled flows  train s  recall@0.5  false alarms@0.5  recall@0.01  false alarms@0.01
     HOIC     0                            9854         2106841                       before (no labels)               0      5.9      1.0000                74       1.0000               3356
     HOIC     0                            9854         2106841                retrained, whole incident             100      5.9      1.0000                98       1.0000               3509
     HOIC     0                            9854         2106841                retrained, whole incident            1000      6.0      1.0000                96       1.0000               3412
     HOIC     0                            9854         2106841                retrained, whole incid

## 5. Findings

6 November 2026, from the outputs above. The app's models are seed 0; the retraining runs use seeds 0 and 1. Recall at 0.5
recomputed here equals the value stored with each of the twelve models, and every model's metadata names the relabelled
20 Feb file, so the splits are the training script's and the models are the retrained ones.

**1. Data.** 9,880,087 flows after cleaning: Benign 7,182,913, HOIC 2,116,841, LOIC-HTTP 575,506, LOIC-UDP 4,827
(1,499 on 20 Feb, 3,328 on 21 Feb). LOIC-UDP is the smallest class by far in flows because each flood flow lasts the
full 120 s timeout and holds about 130,000 packets; in packets it is the largest attack in the data.

**1b. Where the benign flows sit.** On 20 Feb, 900,104 benign flows fall inside the LOIC-HTTP hour but only 135 go to
the victim, and those 135 look like LOIC-HTTP in the destination windows (median 1,531 flows to the same destination,
10 sources) with 0 flows from the same source. The per_day 70 % point of 20 Feb is 14:42 (with the published 12-hour
stamps it was a different set of rows), so every attack of both days sits in the training part of the benign timeline.
On 21 Feb, as in update 5, 291 of the 474 benign flows inside the HOIC hour go to the attacked server and look like
HOIC in the destination windows (6,998 flows, 11 sources) and are told apart only by the source window (0 against 710).

**2. Threshold.**
- XGBoost, the tools it knows (all tools): 8 false alarms among 300,000 benign at 0.5 and 257 (0.09 %) at 0.01; LOIC-UDP
  recall 0.989 at 0.5, 0.996 at 0.01, HOIC and LOIC-HTTP 1.000 throughout.
- XGBoost, unseen HOIC: 0.9999 at every threshold from 0.7 to 0.001; 74 false alarms among 2.15 million at 0.5, 769 at
  0.05, 3,356 at 0.01.
- XGBoost, unseen LOIC-HTTP: 0.0097 at 0.5, 0.48 from 0.3 to 0.1, **0.96 at 0.05 and 0.01** (159 and 1,076 false alarms,
  0.007 % and 0.05 %), 0.98 at 0.001. The attack's median score is 0.096 and its 90th percentile 0.40: below the 0.5
  line, but far above the benign 99.9th percentile (0.0047). The model ranks the attack right and scores it low.
- XGBoost, unseen LOIC-UDP: **0.000 at every threshold down to 0.01** (0.0002 at 0.001). Its median score is 8e-7, the
  same as an ordinary benign flow. Update 5 reported 0.52 "with a ceiling"; that was not a held-out result, because the
  797 flood flows of 20 Feb were in the training set under the LOIC-HTTP label. With the labels fixed there is no
  transfer to an unseen UDP flood (see 3).
- Random Forest: unseen HOIC 0.755 at 0.5 and 0.9999 at 0.3 (570 false alarms); unseen LOIC-HTTP 0.48 at 0.5 and 0.99 at
  0.2 (384). Its scale is a leaf share, not XGBoost's sigmoid of log-odds, so the same number means something else.
- Linear SVM: 0.5 is its own boundary; unseen LOIC-HTTP 0.97 at 0.5 (439 false alarms); below 0.5 its false alarms
  explode (0.5 % at 0.3, 5 % at 0.2). Unseen LOIC-UDP 0.000 at any usable threshold.
- **2b, one threshold per model.** The rule chosen from the benign side only (lowest tested threshold with false alarms
  within 0.1 %) gives: all-tools XGBoost 0.01 (0.09 %, recall 1.000); unseen-LOIC-HTTP XGBoost 0.01 (0.05 %, 0.961);
  unseen-HOIC XGBoost 0.05 (0.04 %); the forests 0.05–0.1; the SVMs 0.5; every by-file and attack-period model 0.5
  because even 0.5 is above the budget (their cards say so). For unseen LOIC-UDP the rule still yields 0.000, which is
  the honest number. The app applies this rule when a model is loaded.

**3. Top features.**
- All-tools XGBoost: flows to the same destination IP (F1) moves the score most (mean |SHAP| 12.0 log-odds), then
  distinct sources to the same destination (F2, 2.0), the forward/backward ratio (F4, 1.1), flows from the same source
  (F3, 0.76), timing regularity (F6, 0.68), no-reply (0.44) and empty payload (0.23).
- Unseen HOIC: the 80 flows missed at the model's default (0.05) are the first seconds of the attack (20 flows to the
  destination instead of 7,069).
- Unseen LOIC-HTTP at 0.01: the 22,743 missed flows have **12 distinct sources instead of 10** (F2 −4.4): the model trained
  on HOIC (11) and benign traffic (4) never saw 12. The split points show the size mismatch (3b): that model's upper splits
  sit at 3,961 flows to the destination, 12 sources and 436 flows per source, learned from HOIC (7,068 / 11 / 710);
  LOIC-HTTP (1,530 / 10 / 154) falls below all three, in a range that the busiest 1 % of 20 Feb benign flows also reach
  (99th percentile 1,631 flows, 122 sources, 353 per source). Transfer worked from the smaller flood to the bigger one
  (HOIC held out) and not the other way round.
- Unseen LOIC-UDP: every flow is missed. F1 = 5 pushes −8.8, and even the ratio of 33,792 pushes towards benign (−1.7):
  the model's ratio splits (0.5–2.1) were learned from HOIC and LOIC-HTTP, which are answered, and the only unanswered
  high-ratio flows it saw were benign scans. "No reply" alone is not an attack signal on this network.

**4. Retraining.** HOIC: 1.000 before any label and after; retraining moves the false alarms by a handful (61 to 98 of
2.15 million at 0.5) and the quick update doubles them (155–158 at 0.5; 7,558–9,738 at 0.01). LOIC-HTTP is now the rescue
case at 0.5: 0.002 before (0.960 at 0.01); 100 whole-incident labels change nothing; **1,000 raise it to 0.96 at 0.5**;
10,000 to 0.971 at 0.5 and 0.995 at 0.01 with 28–35 false alarms. Labelling **only the alerts** (3,766–4,498 flagged among
the first 10,000) reaches only 0.73–0.74 at 0.5 and 0.78–0.86 at 0.01: the SOC loop must label the whole incident. The
quick update (+50 trees) reaches 0.96 at 0.5 but with 153–183 false alarms, and 0.9999 at 0.01 with 10,400–10,650
(0.5 %). A full retraining took about 6 s (HOIC) or 12 s (LOIC-HTTP) on 3.1 million rows. Rule: label the whole incident,
retrain fully, check the new model on a held-out benign set before it replaces the old one.

**Limits.** Two days of one network; ten attacking machines (F2 learned a count, not "many"); the victim server is nearly
idle outside the attacks (median 8 flows per 10 s on 20 Feb, 0 on 21 Feb); the 21 Feb benign flows are 1,982 admin-type
flows; LOIC-UDP is 4,827 rows of which 2,300 are the server's ICMP replies. The window features learned absolute sizes,
so a smaller attack on a busier network reads as normal; a relative version (flows now against the server's own
baseline) is the natural next step. A busy legitimate server or a flash crowd was not available to test.